# 🔬 Tutorial de pkgxray
## Analiza paquetes de PyPI antes de instalarlos

Este notebook demuestra cómo usar **pkgxray** para detectar comportamiento
sospechoso en paquetes Python antes de instalarlos.

In [12]:
!pip install pkgxray

  Cloning https://github.com/maip-fred/pkgxray.git to /tmp/pip-req-build-jgqqthl5
  Running command git clone --filter=blob:none --quiet https://github.com/maip-fred/pkgxray.git /tmp/pip-req-build-jgqqthl5
  Resolved https://github.com/maip-fred/pkgxray.git to commit 3bd3f0cdee75320d7880e2b028c94fc04b6b75a6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## ¿Qué es pkgxray?

Cuando haces `pip install un-paquete`, confías en que ese código no va a:
- Robar tus variables de entorno (API keys, tokens)
- Abrir conexiones de red ocultas
- Ejecutar código arbitrario
- Leer archivos sensibles de tu sistema

**pkgxray** descarga el paquete **sin instalarlo**, analiza el código fuente con AST,
y te dice qué patrones sospechosos encontró.

## 1. Uso básico desde la línea de comandos

In [ ]:
!pkgxray scan numpy

Se truncaron las últimas líneas 5000 del resultado de transmisión.
│            │                  │                                │        │ p… │
│            │                  │                                │        │ e… │
│            │                  │                                │        │ en │
│            │                  │                                │        │ el │
│            │                  │                                │        │ s… │
│            │                  │                                │        │ de │
│            │                  │                                │        │ a… │
│ MEDIUM     │ env_access       │ son/run_meson_command_tests.py │     64 │ Se │
│            │                  │                                │        │ d… │
│            │                  │                                │        │ a… │
│            │                  │                                │        │ a  │
│            │                  │         

## 2. Uso desde Python (API)

In [13]:
from pkgxray import scan

result = scan("attrs")
print(f"Paquete: {result.package_name}")
print(f"Versión: {result.version}")
print(f"Puntaje de riesgo: {result.risk_score}/100")
print(f"Nivel de riesgo: {result.risk_level}")
print(f"Archivos analizados: {result.files_analyzed}")
print(f"Hallazgos totales: {len(result.findings)}")

Paquete: attrs
Versión: 26.1.0
Puntaje de riesgo: 100/100
Nivel de riesgo: CRITICAL
Archivos analizados: 55
Hallazgos totales: 24


## 3. Explorando los hallazgos en detalle

In [ ]:
for finding in result.findings[:10]:  # primeros 10 hallazgos
    print(f"[{finding.severity.value.upper()}] {finding.analyzer_name}")
    print(f"  Archivo: {finding.filename}:{finding.line_number}")
    print(f"  Descripción: {finding.description}")
    print(f"  Código: {finding.code_snippet[:100]}")
    print()

[MEDIUM] network
  Archivo: requests-2.33.1/src/requests/adapters.py:10
  Descripción: Se detectó importación del módulo de red 'socket'
  Código: import socket  # noqa: F401

[HIGH] network
  Archivo: requests-2.33.1/src/requests/adapters.py:645
  Descripción: Se detectó llamada de red 'urlopen()'
  Código: resp = conn.urlopen(

[HIGH] network
  Archivo: requests-2.33.1/src/requests/auth.py:133
  Descripción: Se detectó llamada de red 'get()'
  Código: qop = self._thread_local.chal.get("qop")

[HIGH] network
  Archivo: requests-2.33.1/src/requests/auth.py:134
  Descripción: Se detectó llamada de red 'get()'
  Código: algorithm = self._thread_local.chal.get("algorithm")

[HIGH] network
  Archivo: requests-2.33.1/src/requests/auth.py:135
  Descripción: Se detectó llamada de red 'get()'
  Código: opaque = self._thread_local.chal.get("opaque")

[HIGH] network
  Archivo: requests-2.33.1/src/requests/auth.py:258
  Descripción: Se detectó llamada de red 'get()'
  Código: s_auth = r.headers.g

## 4. Generar reporte en JSON

In [ ]:
from pkgxray.reporter import generate_report

json_report = generate_report(result, format="json")
print(json_report[:2000])  # primeros 2000 caracteres

{
  "package_name": "requests",
  "version": "2.33.1",
  "scan_date": "2026-04-14T20:11:50.305660+00:00",
  "risk_score": 100,
  "risk_level": "CRITICAL",
  "files_analyzed": 36,
  "summary": {
    "low": 0,
    "medium": 50,
    "high": 219,
    "critical": 1,
    "total": 270
  },
  "findings": [
    {
      "severity": "medium",
      "description": "Se detect\u00f3 importaci\u00f3n del m\u00f3dulo de red 'socket'",
      "filename": "requests-2.33.1/src/requests/adapters.py",
      "line_number": 10,
      "code_snippet": "import socket  # noqa: F401",
      "analyzer_name": "network"
    },
    {
      "severity": "high",
      "description": "Se detect\u00f3 llamada de red 'urlopen()'",
      "filename": "requests-2.33.1/src/requests/adapters.py",
      "line_number": 645,
      "code_snippet": "resp = conn.urlopen(",
      "analyzer_name": "network"
    },
    {
      "severity": "high",
      "description": "Se detect\u00f3 llamada de red 'get()'",
      "filename": "requests-2

## 5. Analizar otro paquete y comparar

In [ ]:
result2 = scan("flask")
print(f"flask    - Puntaje: {result2.risk_score}/100 ({result2.risk_level})")
print(f"flask    - Hallazgos: {len(result2.findings)}")
print()
print(f"requests - Puntaje: {result.risk_score}/100 ({result.risk_level})")
print(f"requests - Hallazgos: {len(result.findings)}")

## 6. Resumen y análisis por tipo de hallazgo

In [ ]:
from collections import Counter

analyzer_counts = Counter(f.analyzer_name for f in result.findings)
severity_counts = Counter(f.severity.value for f in result.findings)

print("Hallazgos por analizador:")
for name, count in analyzer_counts.most_common():
    print(f"  {name}: {count}")

print("\nHallazgos por severidad:")
for sev, count in severity_counts.most_common():
    print(f"  {sev}: {count}")

Hallazgos por analizador:
  network: 245
  env_access: 15
  filesystem: 8
  dynamic_imports: 2

Hallazgos por severidad:
  high: 219
  medium: 50
  critical: 1


## 7. Generar reporte HTML

In [ ]:
html_report = generate_report(result, format="html")
from IPython.display import HTML, display
display(HTML(html_report))

Severidad,Analizador,Archivo,Línea,Descripción,Fragmento
MEDIUM,network,requests-2.33.1/src/requests/adapters.py,10,Se detectó importación del módulo de red 'socket',import socket # noqa: F401
HIGH,network,requests-2.33.1/src/requests/adapters.py,645,Se detectó llamada de red 'urlopen()',resp = conn.urlopen(
HIGH,network,requests-2.33.1/src/requests/auth.py,133,Se detectó llamada de red 'get()',"qop = self._thread_local.chal.get(""qop"")"
HIGH,network,requests-2.33.1/src/requests/auth.py,134,Se detectó llamada de red 'get()',"algorithm = self._thread_local.chal.get(""algorithm"")"
HIGH,network,requests-2.33.1/src/requests/auth.py,135,Se detectó llamada de red 'get()',"opaque = self._thread_local.chal.get(""opaque"")"
HIGH,network,requests-2.33.1/src/requests/auth.py,258,Se detectó llamada de red 'get()',"s_auth = r.headers.get(""www-authenticate"", """")"
MEDIUM,network,requests-2.33.1/src/requests/compat.py,81,Se detectó importación desde el módulo de red 'urllib.parse',from urllib.parse import (
MEDIUM,network,requests-2.33.1/src/requests/compat.py,93,Se detectó importación desde el módulo de red 'urllib.request',from urllib.request import (
HIGH,dynamic_imports,requests-2.33.1/src/requests/compat.py,36,Se detectó llamada a importlib.import_module() con argumento dinámico,chardet = importlib.import_module(lib)
HIGH,network,requests-2.33.1/src/requests/cookies.py,148,Se detectó llamada de red 'get()',"return r.get_new_headers().get(""Cookie"")"


## Conclusión

**pkgxray** te permite auditar paquetes de PyPI antes de instalarlos,
detectando patrones como ejecución dinámica de código, conexiones de red,
acceso a variables de entorno, ofuscación y más.

### Recursos
- [Repositorio en GitHub](https://github.com/maip-fred/pkgxray)
- [Paquete en PyPI](https://pypi.org/project/pkgxray/)
- `pip install pkgxray`

## 8. Comparativa de paquetes: limpio vs sospechoso

In [14]:
from pkgxray import scan

# Comparativa: paquetes con diferentes perfiles de riesgo
paquetes = [
    ("more-itertools", "utilidades puras"),
    ("attrs",          "utilidades puras"),
    ("click",          "CLI / E/S"),
    ("requests",       "HTTP"),
    ("paramiko",       "SSH"),
]

print(f"{'Paquete':<22} {'Tipo':<20} {'Score':>6}  {'Nivel'}")
print("-" * 60)
for pkg, tipo in paquetes:
    r = scan(pkg)
    print(f"{pkg:<22} {tipo:<20} {r.risk_score:>5}/100  {r.risk_level}")

Paquete                Tipo                  Score  Nivel
------------------------------------------------------------
more-itertools         utilidades puras        52/100  HIGH
attrs                  utilidades puras       100/100  CRITICAL
click                  CLI / E/S              100/100  CRITICAL
requests               HTTP                   100/100  CRITICAL
paramiko               SSH                    100/100  CRITICAL
